# 04_inference_demo.ipynb — Inference & Alerting Pipeline

> **Project:** SentinelFatal2 — Foundation Model for fetal distress prediction from CTG  
> **Source:** arXiv:2601.06149v1, Section II-F, Figure 5  
> **SSOT:** `docs/work_plan.md`, Part ו, Step 5  
> **Agent:** Agent 5 (demo) + Agent 6 (LR training)

## Notebook structure

| Cell | Purpose |
|------|----------|
| 1 | Setup: paths, seed, sys.path (+ Colab Drive mount) |
| 2 | GPU check |
| 3 | Imports + constants |
| 4 | Load fine-tuned model |
| 5 | Pick demo recording |
| 6 | Stage 1: sliding-window inference + V5.1 |
| 7 | Stage 2a: alert segments + V5.2 |
| 8 | Stage 2b: alert features + V5.3 |
| 9 | V5.4 + V5.6 code checks |
| 10 | V5.5 LR checkpoint validation |
| 11 | Visualization |
| 12 | **Full LR training** (Agent 6) |
| 13 | Summary

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os, random
import numpy as np
import torch

# ── Colab: mount Drive and cd to project ──────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/SentinelFatal2'
    if not os.path.isdir(PROJECT_ROOT):
        print('Project folder not found on Drive — cloning from GitHub...')
        os.system(f'git clone https://github.com/ArielShamay/SentinelFatal2.git "{PROJECT_ROOT}"')
    os.chdir(PROJECT_ROOT)
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
else:
    from pathlib import Path as _P
    PROJECT_ROOT = str(_P(os.path.abspath('')).parent
                       if _P(os.path.abspath('')).name == 'notebooks'
                       else _P(os.path.abspath('')))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

print(f'Project root : {os.getcwd()}')
print(f'Python       : {sys.version}')
print(f'PyTorch      : {torch.__version__}')
print(f'[seed] All seeds set to {SEED}')

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    DEVICE = torch.device('cpu')
    print('No GPU found -- running on CPU')

print(f'torch {torch.__version__}  |  device: {DEVICE}')

In [ ]:
# ── Cell 3: Imports + constants ────────────────────────────────────────────────
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from src.model.patchtst import PatchTST, load_config
from src.model.heads import ClassificationHead
from src.inference.sliding_window import (
    INFERENCE_STRIDE_REPRO,
    INFERENCE_STRIDE_RUNTIME,
    inference_recording,
)
from src.inference.alert_extractor import (
    ALERT_THRESHOLD,
    extract_alert_segments,
    compute_alert_features,
    ZERO_FEATURES,
)

print(f'INFERENCE_STRIDE_REPRO   = {INFERENCE_STRIDE_REPRO}')
print(f'INFERENCE_STRIDE_RUNTIME = {INFERENCE_STRIDE_RUNTIME}')
print(f'ALERT_THRESHOLD          = {ALERT_THRESHOLD}')

In [ ]:
# ── Cell 4: Load fine-tuned model ─────────────────────────────────────────────
pr = Path(PROJECT_ROOT)
CONFIG_PATH = pr / 'config' / 'train_config.yaml'
CKPT_PATH   = pr / 'checkpoints' / 'finetune' / 'best_finetune.pt'

config = load_config(CONFIG_PATH)

model = PatchTST(config)
d_in = (
    config['data']['n_patches']
    * config['model']['d_model']
    * config['data']['n_channels']
)  # 73 * 128 * 2 = 18688
model.replace_head(ClassificationHead(d_in=d_in))

state = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state, strict=False)
model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded from {CKPT_PATH}')
print(f'Total parameters: {n_params:,}')
print(f'Device: {DEVICE}')

In [ ]:
# ── Cell 5: Pick demo recording ───────────────────────────────────────────────
pr = Path(PROJECT_ROOT)
TRAIN_CSV     = pr / 'data' / 'splits' / 'train.csv'
PROCESSED_DIR = pr / 'data' / 'processed' / 'ctu_uhb'

df = pd.read_csv(TRAIN_CSV, dtype={'id': str, 'target': int})
demo_row = df.iloc[0]
REC_ID    = demo_row['id']
REC_LABEL = int(demo_row['target'])
NPY_PATH  = PROCESSED_DIR / f'{REC_ID}.npy'

print(f'Recording : {REC_ID}')
print(f'Label     : {REC_LABEL}  (1=acidemia, 0=normal)')
print(f'NPY path  : {NPY_PATH}')
assert NPY_PATH.exists(), f'Missing: {NPY_PATH}'

signal = np.load(NPY_PATH, mmap_mode='r')  # (2, T)
print(f'Signal shape: {signal.shape}  (2=FHR+UC, T=samples)')
print(f'Duration    : {signal.shape[1] / 4 / 60:.1f} min @ 4 Hz')

In [ ]:
# ── Stage 1: Sliding-window inference (RUNTIME stride for demo speed) ─────────
# NOTE: Official evaluation always uses INFERENCE_STRIDE_REPRO=1 (Stage 7)
DEMO_STRIDE = INFERENCE_STRIDE_RUNTIME  # 60 samples = 15s steps (demo only)

print(f'Running inference with stride={DEMO_STRIDE} (runtime/demo mode)...')
scores = inference_recording(model, signal, stride=DEMO_STRIDE, device=str(DEVICE))

print(f'Total windows scored: {len(scores)}')
starts  = [s for s, _ in scores]
probs   = [p for _, p in scores]
print(f'Score range: [{min(probs):.4f}, {max(probs):.4f}]')

# V5.1 — validate return type
assert all(isinstance(s, int)   for s, _ in scores), 'start_sample must be int'
assert all(0.0 <= p <= 1.0      for _, p in scores), 'score must be in [0, 1]'
print('[V5.1] PASS: inference_recording() returns list of (int, float), score in [0,1]')

In [ ]:
# --- Stage 2a: Extract alert segments ---
segments = extract_alert_segments(scores, threshold=ALERT_THRESHOLD)

print(f'Alert segments found: {len(segments)}')
for i, (start_s, end_s, seg_scores) in enumerate(segments):
    dur_sec = len(seg_scores) * DEMO_STRIDE / 4.0
    print(f'  Segment {i+1}: start={start_s}, end={end_s}, '
          f'n_windows={len(seg_scores)}, max_score={max(seg_scores):.4f}, '
          f'duration~{dur_sec:.0f}s')

# V5.2 — all segments have scores > threshold
for start_s, end_s, seg_scores in segments:
    assert all(s > ALERT_THRESHOLD for s in seg_scores), (
        f'Segment score below threshold {ALERT_THRESHOLD}'
    )
print(f'[V5.2] PASS: extract_alert_segments() returns segments with threshold={ALERT_THRESHOLD}')

In [ ]:
# --- Stage 2b: Compute alert features ---
if segments:
    longest = max(segments, key=lambda seg: len(seg[2]))
    _, _, seg_scores = longest
    feats = compute_alert_features(seg_scores, inference_stride=DEMO_STRIDE, fs=4.0)
else:
    feats = dict(ZERO_FEATURES)
    print('[demo] No alert segments -> using ZERO_FEATURES (assumption S10)')

print('Alert features (4 keys):')
for k, v in feats.items():
    print(f'  {k}: {v:.6f}')

# V5.3 — exactly 4 keys
assert len(feats) == 4, f'Expected 4 features, got {len(feats)}'
expected_keys = {'segment_length', 'max_prediction', 'cumulative_sum', 'weighted_integral'}
assert set(feats.keys()) == expected_keys, f'Wrong keys: {set(feats.keys())}'
print('[V5.3] PASS: compute_alert_features() returns dict with exactly 4 keys')

In [ ]:
# ── V5.4: LR trained on Train only (AGW-26 fix: regex I/O check) ─────────────
import re
pr = Path(PROJECT_ROOT)
train_lr_src = (pr / 'src' / 'train' / 'train_lr.py').read_text(encoding='utf-8')

io_violations = re.findall(
    r'(?:read_csv|np\.load|open)\s*\([^)]*(?:test\.csv|val\.csv)',
    train_lr_src,
)
assert len(io_violations) == 0, 'VIOLATION: I/O access to val/test: ' + str(io_violations)
assert 'train.csv' in train_lr_src, 'train.csv not found in train_lr.py'
print('[V5.4] PASS: train_lr.py has no I/O operations on test.csv or val.csv')

# V5.6: stride=REPRO constant used
assert 'INFERENCE_STRIDE_REPRO' in train_lr_src, 'INFERENCE_STRIDE_REPRO must appear in train_lr.py'
print(f'[V5.6] PASS: INFERENCE_STRIDE_REPRO (={INFERENCE_STRIDE_REPRO}) is the training stride in train_lr.py')

In [ ]:
# ── V5.5: LR checkpoint validation (AGW-27 guard) ────────────────────────────
pr = Path(PROJECT_ROOT)
LR_CKPT = pr / 'checkpoints' / 'alerting' / 'logistic_regression.pkl'
if LR_CKPT.exists():
    from src.train.train_lr import validate_lr_checkpoint
    ok, msg = validate_lr_checkpoint(LR_CKPT,
                                     expected_stride=INFERENCE_STRIDE_REPRO,
                                     min_n_train=397)
    assert ok, '[V5.5] FAIL: ' + msg
    print('[V5.5] PASS: ' + msg)
else:
    print(f'[V5.5] SKIP: {LR_CKPT} does not exist yet.')
    print('  Run the LR Training cell below to generate it.')

In [ ]:
# --- Visualization: Score timeline + Alert regions ---
FS = 4  # Hz

time_min = [s / FS / 60.0 for s in starts]  # x-axis in minutes

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# FHR signal
fhr_time  = np.arange(signal.shape[1]) / FS / 60.0
axes[0].plot(fhr_time, signal[0], lw=0.7, color='steelblue', label='FHR')
axes[0].set_ylabel('FHR (normalised)')
axes[0].set_title(f'Recording {REC_ID}  |  label={REC_LABEL} (1=acidemia)')
axes[0].legend(loc='upper right')

# NN score timeline
axes[1].plot(time_min, probs, lw=1.0, color='darkorange', label='P(acidemia)')
axes[1].axhline(ALERT_THRESHOLD, color='red', linestyle='--', lw=1.2,
                label=f'threshold={ALERT_THRESHOLD}')

# Shade alert segments
for seg_i, (start_s, end_s, _) in enumerate(segments):
    x0 = start_s / FS / 60.0
    x1 = end_s   / FS / 60.0
    axes[1].axvspan(x0, x1, color='red', alpha=0.25,
                    label='alert segment' if seg_i == 0 else None)

axes[1].set_xlabel('Time (minutes)')
axes[1].set_ylabel('P(acidemia)')
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()
print(f'Plot shown for recording {REC_ID}  (stride={DEMO_STRIDE}, n_windows={len(scores)})')

In [ ]:
# ── Full LR Training (Agent 6) ────────────────────────────────────────────────
#
# This trains the Stage-2 Logistic Regression on all 441 train recordings.
# Uses stride=INFERENCE_STRIDE_REPRO=1 (maximum precision, slow).
# Requires a valid best_finetune.pt checkpoint.
#
# Set DRY_RUN = True for quick 5-recording test.
# Set DRY_RUN = False for official full-train run.
#
import subprocess

DRY_RUN = True    # <-- change to False for full LR training on Colab

max_rec_flag = ['--max-recordings', '5'] if DRY_RUN else []

cmd = [
    sys.executable, 'src/train/train_lr.py',
    '--config', 'config/train_config.yaml',
    '--device', str(DEVICE),
] + max_rec_flag

print('Command:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print(f'\nLR training script exited with code {result.returncode}')
else:
    if DRY_RUN:
        print('\nDry-run LR training complete (5 recordings)')
    else:
        print('\nFull LR training complete (all 441 train recordings)')

    # Validate the new checkpoint
    LR_CKPT = Path(PROJECT_ROOT) / 'checkpoints' / 'alerting' / 'logistic_regression.pkl'
    if LR_CKPT.exists():
        from src.train.train_lr import validate_lr_checkpoint
        ok, msg = validate_lr_checkpoint(LR_CKPT,
                                         expected_stride=INFERENCE_STRIDE_REPRO,
                                         min_n_train=5 if DRY_RUN else 397)
        print(f'Checkpoint validation: {msg}')
    else:
        print(f'WARNING: {LR_CKPT} was not created!')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=== Inference & Alerting Pipeline Summary ===')
print(f'  Recording   : {REC_ID} (label={REC_LABEL})')
print(f'  Windows     : {len(scores)} (stride={DEMO_STRIDE})')
print(f'  Alert segs  : {len(segments)}')
print(f'  Features    : {feats}')
lr_ckpt = Path(PROJECT_ROOT) / 'checkpoints' / 'alerting' / 'logistic_regression.pkl'
print(f'  LR ckpt     : {"EXISTS" if lr_ckpt.exists() else "MISSING"}')
print('[G5.4] PASS: scores list produced, segments extracted, features computed')